# Layers
Input → Embedding → LSTM → Dropout → Hidden Dense → Dropout → Output

# 1.Install Libraries

In [1]:
!pip install datasets -q

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf

from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from sklearn.metrics import classification_report, confusion_matrix

2026-05-16 06:10:47.722272: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778911848.134315      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778911848.250971      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778911849.234895      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778911849.234947      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778911849.234950      57 computation_placer.cc:177] computation placer alr

# 2. Load Datasets

In [3]:
dataset=load_dataset("cardiffnlp/tweet_eval","offensive")
train_data=dataset["train"]
val_data=dataset["validation"]
test_data=dataset["test"]
print(train_data[[0]])

README.md: 0.00B [00:00, ?B/s]

offensive/train-00000-of-00001.parquet:   0%|          | 0.00/1.02M [00:00<?, ?B/s]

offensive/test-00000-of-00001.parquet:   0%|          | 0.00/93.7k [00:00<?, ?B/s]

offensive/validation-00000-of-00001.parq(…):   0%|          | 0.00/122k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11916 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/860 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1324 [00:00<?, ? examples/s]

{'text': ['@user Bono... who cares. Soon people will understand that they gain nothing from following a phony celebrity. Become a Leader of your people instead or help and support your fellow countrymen.'], 'label': [0]}


In [4]:
print(dataset)
print(train_data[0:2])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 11916
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 860
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1324
    })
})
{'text': ['@user Bono... who cares. Soon people will understand that they gain nothing from following a phony celebrity. Become a Leader of your people instead or help and support your fellow countrymen.', '@user Eight years the republicans denied obama’s picks. Breitbarters outrage is as phony as their fake president.'], 'label': [0, 1]}


# 3. Convert dataset to lists

In [5]:
print(train_data['label'])

Column([0, 1, 0, 0, 1, ...])


In [6]:
X_train=train_data['text']
y_train=np.array(train_data['label'])

X_val=val_data['text']
y_val=np.array(val_data['label'])

X_test=test_data['text']
y_test=np.array(test_data['label'])

# 4. Tokenize text

In [7]:
lengths=[len(text.split()) for text in X_train]
print("Max length",np.max(lengths))
print("95th percentile:", np.percentile(lengths, 95))

Max length 103
95th percentile: 51.0


In [8]:
MAX_WORDS=10000
MAX_LEN=60

tokenizer=Tokenizer(num_words=MAX_WORDS,oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq=tokenizer.texts_to_sequences(X_train)
X_val_seq=tokenizer.texts_to_sequences(X_val)
X_test_seq=tokenizer.texts_to_sequences(X_test)


# 5. Padding

In [9]:
X_train_pad=pad_sequences(X_train_seq,maxlen=MAX_LEN,padding="post",truncating="post")
X_val_pad=pad_sequences(X_val_seq,maxlen=MAX_LEN,padding="post",truncating="post")
X_test_pad=pad_sequences(X_test_seq,maxlen=MAX_LEN,padding="post",truncating="post")


# 6. Build LSTM model

In [10]:
model=Sequential([
    Embedding(input_dim=MAX_WORDS,output_dim=128),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(32,activation="relu"),
    Dropout(0.3),
    Dense(1,activation="sigmoid")
])
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

I0000 00:00:1778911896.076324      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778911896.082764      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# 7. Train model

In [11]:
history=model.fit(X_train_pad,
                  y_train,
                  validation_data=(X_val_pad,y_val),
                  epochs=5,
                  batch_size=32,
                 )

Epoch 1/5


I0000 00:00:1778911901.501504     157 cuda_dnn.cc:529] Loaded cuDNN version 91002


373/373 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - accuracy: 0.6660 - loss: 0.6268 - val_accuracy: 0.7492 - val_loss: 0.5305
Epoch 2/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.8104 - loss: 0.4424 - val_accuracy: 0.7696 - val_loss: 0.5301
Epoch 3/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.8727 - loss: 0.3269 - val_accuracy: 0.7576 - val_loss: 0.5632
Epoch 4/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9105 - loss: 0.2367 - val_accuracy: 0.7387 - val_loss: 0.6536
Epoch 5/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9412 - loss: 0.1686 - val_accuracy: 0.7447 - val_loss: 0.7714


# 8. Prediction & Error analysis

In [12]:
y_prob=model.predict(X_test_pad).ravel()#flatten the predicted output
y_pred=(y_prob>=0.5).astype(int)
print(classification_report(y_test,y_pred, target_names=["non-offensive", "offensive"]))
print(confusion_matrix(y_test,y_pred))

27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
               precision    recall  f1-score   support

non-offensive       0.85      0.82      0.84       620
    offensive       0.58      0.62      0.60       240

     accuracy                           0.77       860
    macro avg       0.71      0.72      0.72       860
 weighted avg       0.77      0.77      0.77       860

[[511 109]
 [ 91 149]]


The model achieves reasonable performance, but still struggles with the offensive class.

# 9. Add neutral class

In [13]:
def predict_with_uncertainty(prob):
    if(prob>=0.6):
        return "offensive"
    elif prob<=0.4:
        return "non-offensive"
    else:
        return "neutral/uncertain"

In [14]:
uncertain_preds=[predict_with_uncertainty(p) for p in y_prob]
result_df=pd.DataFrame({
    "text":X_test,
    "true_label":y_test,
    "toxic_probability":y_prob,
    "prediction": uncertain_preds
})
result_df.head(20)

,text,true_label,toxic_probability,prediction
0,#ibelieveblaseyford is liar she is fat ugly li...,1,0.996405,offensive
1,@user @user @user I got in a pretty deep debat...,0,0.991078,offensive
2,"...if you want more shootings and more death, ...",0,0.387494,non-offensive
3,Angels now have 6 runs. Five of them have come...,0,0.001993,non-offensive
4,#Travel #Movies and Unix #Fortune combined Vi...,0,0.015729,non-offensive
5,#naturephotography. #nature. #birds #wild in #...,0,0.002650,non-offensive
6,#i2 This speaks for itself (NSFW),0,0.043163,non-offensive
7,#WomenWednesday: Meet multiple-award-winning d...,0,0.366820,non-offensive
8,". a grown ass woman, probably 10 years older t...",1,0.270522,non-offensive
9,#RAP is a form of ART! Used to express yoursel...,0,0.018974,non-offensive


In [15]:
model.save("/kaggle/working/saved_bilstm.keras")

In [16]:
import pickle

with open("/kaggle/working/bilstm_tokenizer.pkl","wb") as f:
    pickle.dump(tokenizer,f)

In [17]:
#save max length
import json
bilstm_config={
    "max_len":60
}
with open("/kaggle/working/bilstm_config.json","w") as f:
    json.dump(bilstm_config,f)

In [18]:
!zip -r /kaggle/working/saved_bilstm_files.zip \
/kaggle/working/saved_bilstm.keras \
/kaggle/working/bilstm_tokenizer.pkl \
/kaggle/working/bilstm_config.json

  adding: kaggle/working/saved_bilstm.keras (deflated 6%)
  adding: kaggle/working/bilstm_tokenizer.pkl (deflated 54%)
  adding: kaggle/working/bilstm_config.json (stored 0%)


In [19]:
from IPython.display import FileLink
FileLink("/kaggle/working/saved_bilstm_files.zip")

/kaggle/working/saved_bilstm_files.zip

# Transformer

# 10. install libraries

In [20]:
!pip install transformers evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


# 11. Load dataset

In [21]:
from datasets import load_dataset
dataset2=load_dataset("cardiffnlp/tweet_eval","offensive")

train_dataset2=dataset2["train"]
val_dataset2=dataset2["validation"]
test_dataset2=dataset2["test"]

print(dataset2)
print(train_dataset2[0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 11916
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 860
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1324
    })
})
{'text': '@user Bono... who cares. Soon people will understand that they gain nothing from following a phony celebrity. Become a Leader of your people instead or help and support your fellow countrymen.', 'label': 0}


# 12. Tokenize for Transformer

In [22]:
from transformers import AutoTokenizer

MODEL_NAME="distilbert-base-uncased"
tokenizer2=AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer2(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )
tokenized_dataset=dataset2.map(tokenize_function,batched=True)

print(tokenized_dataset)
print(tokenized_dataset["train"][0])


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/11916 [00:00<?, ? examples/s]

Map:   0%|          | 0/860 [00:00<?, ? examples/s]

Map:   0%|          | 0/1324 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 11916
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 860
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1324
    })
})
{'text': '@user Bono... who cares. Soon people will understand that they gain nothing from following a phony celebrity. Become a Leader of your people instead or help and support your fellow countrymen.', 'label': 0, 'input_ids': [101, 1030, 5310, 23648, 1012, 1012, 1012, 2040, 14977, 1012, 2574, 2111, 2097, 3305, 2008, 2027, 5114, 2498, 2013, 2206, 1037, 6887, 16585, 8958, 1012, 2468, 1037, 3003, 1997, 2115, 2111, 2612, 2030, 2393, 1998, 2490, 2115, 3507, 2406, 3549, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids':

In [23]:
print(tokenized_dataset.column_names)

{'train': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'], 'test': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'], 'validation': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']}


# 13. Prepare labels

In [24]:
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","labels"]
)

# 14. Build Transformer model

In [25]:
from transformers import AutoModelForSequenceClassification

model2=AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# 15. Metrics

In [26]:
import numpy as np
from sklearn.metrics import accuracy_score,precision_recall_fscore_support
def compute_metrics(eval_pred):
    logits,labels=eval_pred
    preds=np.argmax(logits,axis=1)
    precision,recall,f1,_=precision_recall_fscore_support(
        labels,
        preds,
        average="macro"
    )
    acc=accuracy_score(labels,preds)

    return{
        "accuracy":acc,
        "macro_precision":precision,
        "macro_recall":recall,
        "macro_f1":f1
    }

# 16. Train Transformer

In [27]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./transformer_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none"
)

trainer = Trainer(
    model=model2,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,No log,0.899963,0.787009,0.767129,0.783309,0.772534
2,0.889322,0.892808,0.790785,0.769162,0.777507,0.772749
3,0.700130,0.920764,0.798338,0.778209,0.772550,0.775172


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1119, training_loss=0.7732941164727505, metrics={'train_runtime': 169.7828, 'train_samples_per_second': 210.551, 'train_steps_per_second': 6.591, 'total_flos': 591930570894336.0, 'train_loss': 0.7732941164727505, 'epoch': 3.0})

# 17. Evaluate on test set

In [28]:
test_result=trainer.evaluate(tokenized_dataset["test"])
print(test_result)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.7625204920768738, 'eval_accuracy': 0.85, 'eval_macro_precision': 0.8193798449612403, 'eval_macro_recall': 0.7976478494623656, 'eval_macro_f1': 0.8072536159492681, 'eval_runtime': 1.3814, 'eval_samples_per_second': 622.575, 'eval_steps_per_second': 19.546, 'epoch': 3.0}


In [29]:
from sklearn.metrics import classification_report,confusion_matrix

pred_output=trainer.predict(tokenized_dataset["test"])
logits=pred_output.predictions
y_pred2=np.argmax(logits,axis=1)
y_true=pred_output.label_ids
print(classification_report(
    y_true,
    y_pred2,
    target_names=["non-offensive","offensive"]
))

print(confusion_matrix(y_true,y_pred2))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


               precision    recall  f1-score   support

non-offensive       0.88      0.92      0.90       620
    offensive       0.76      0.68      0.72       240

     accuracy                           0.85       860
    macro avg       0.82      0.80      0.81       860
 weighted avg       0.85      0.85      0.85       860

[[568  52]
 [ 77 163]]


# 18. Save Transformer model

In [30]:
trainer.save_model("./saved_transformer")
tokenizer2.save_pretrained("./saved_transformer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_transformer/tokenizer_config.json',
 './saved_transformer/tokenizer.json')

In [31]:
!zip -r saved_transformer.zip ./saved_transformer

  adding: saved_transformer/ (stored 0%)
  adding: saved_transformer/tokenizer.json (deflated 71%)
  adding: saved_transformer/tokenizer_config.json (deflated 42%)
  adding: saved_transformer/config.json (deflated 49%)
  adding: saved_transformer/model.safetensors (deflated 8%)
  adding: saved_transformer/training_args.bin (deflated 53%)


In [32]:
from IPython.display import FileLink

FileLink("/kaggle/working/saved_transformer.zip")

/kaggle/working/saved_transformer.zip